In [ ]:
recordingID = '221028_0_1'


from google.colab import drive
drive.mount('/content/drive')

import torch

if torch.cuda.is_available():
    print('\033[1;31mGPU accessiable. Use GPU for computation.\033[0m')
    gpu_id = torch.cuda.current_device()
    total_memory = torch.cuda.get_device_properties(gpu_id).total_memory/1024/1024
    alloc_memory = torch.cuda.memory_allocated(0)/1024/1024
    print('GPU ID: ', gpu_id, '|', torch.cuda.get_device_name(), \
          '| Memory: {:.0f} MB'.format(total_memory))
    ! nvcc --version
    print('PyTorch version: ', torch.__version__)
else:
    print('\033[1;31mNo GPU support. Please enable GPUs for the notebook:\033[0m')
    print(' 1. Navigate to Edit → Notebook Settings')
    print(' 2. Select GPU from the Hardware Accelerator drop-down')

!pip install deepcad

!pip install mat73

import os
import glob
import shutil

import numpy as np
from skimage import io
import tifffile as tif
import mat73

import scipy.io
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

from deepcad.train_collection import training_class
from deepcad.test_collection import testing_class
from deepcad.movie_display import display, display_img
from deepcad.utils import get_first_filename

import random
from tqdm import tqdm
from scipy import signal
import datetime


class BgRemover:


    def __init__(self, upstream_path, img_path, half_wid=12):
        self.uppath = upstream_path
        self.path = img_path
        self.half_wid = half_wid
        self.img = io.imread(img_path).astype('float32')
        self.make_savedir()

    def make_savedir(self):
        working_dir = os.path.dirname(self.uppath)
        saving_dir = os.path.join(working_dir, 'bg_remove')
        if not os.path.exists(saving_dir):
            os.mkdir(saving_dir)
        self.saving_dir = saving_dir
        self.file_head = self.path.split('.')[0].split('/')[-1]

    def draw_bg(self):
        half_wid = self.half_wid
        wid = 2*half_wid
        kernel = np.ones(wid)/wid
        template = np.mean(self.img, axis=0)
        bg_ind = []
        for line in template:
            tmp = np.convolve(line, kernel, 'valid')
            bg_center = np.argmin(tmp) + half_wid
            bg_ind.append([bg_center-half_wid, bg_center+half_wid])
        self.bg_ind = bg_ind

    def show_bg(self):
        bg_ind = self.bg_ind
        show_bg = np.mean(self.img, axis=0)
        mv = np.max(show_bg)
        for i in range(show_bg.shape[0]):
            show_bg[i, bg_ind[i][0]:bg_ind[i][1]] = mv
        plt.imshow(show_bg)
        plt.savefig(os.path.join(self.saving_dir, self.file_head+'_bg_patch.png'))
        plt.close()

    def remove_bg(self, offset=0):
        bg_ind = self.bg_ind
        out = self.img.copy()
        for ind in range(out.shape[1]):
            patch = self.img[:, ind, :]
            bg_patch = self.img[:, ind, bg_ind[ind][0]:bg_ind[ind][1]]
            bg = bg_patch.mean(axis=-1)
            patch = patch-bg[None].T
            out[:, ind, :] = patch
        out = out + offset # this is to compensate the image so that most of the pixels are above 0
        self.out = out

    def show_spectrum(self, fs=354):
        half_wid = self.half_wid
        y_len, x_len = self.out.shape[1:3]
        test_y = random.randint(0, y_len-1)
        test_x = random.randint(0, x_len-2*half_wid-1) + half_wid

        test_patch = self.img[:, test_y, test_x-half_wid:test_x+half_wid]
        test = test_patch.mean(-1)
        test = test/test.mean()

        f, Pxx_den = signal.periodogram(test, fs)
        plt.semilogy(f, Pxx_den)
        plt.ylim([1e-7, 1])
        plt.savefig(os.path.join(self.saving_dir, self.file_head+'_spectrum_withBG.png'))
        plt.close()

        test_patch = self.out[:, test_y, test_x-half_wid:test_x+half_wid]
        test = test_patch.mean(-1)
        test = test/test.mean()

        f, Pxx_den = signal.periodogram(test, fs)
        plt.semilogy(f, Pxx_den)
        plt.ylim([1e-7, 1])
        plt.savefig(os.path.join(self.saving_dir, self.file_head+'_spectrum_withoutBG.png'))
        plt.close()

    def save_out(self):
        save_name = os.path.join(self.path[:-4] + '.tif')
        io.imsave(save_name, self.out.astype('float'))




working_dir = '/content/drive/MyDrive/'
datasets_path = os.path.join(working_dir, 'stacks')
datasets_path_processing = os.path.join(working_dir, 'denoising_in_progress')
if not os.path.exists(datasets_path_processing):
    os.mkdir(datasets_path_processing)
datasets_path_complete = os.path.join(working_dir, 'denoised')
if not os.path.exists(datasets_path_complete):
    os.mkdir(datasets_path_complete)
stack_filename_template = recordingID + '_stackRaw_mc_.mat'
stack_paths = glob.glob(os.path.join(datasets_path, stack_filename_template))

for f in stack_paths:

  mat = mat73.loadmat(f)
  stack = mat['stackRaw_mc']
  print(stack.shape)
  stack = np.moveaxis(stack, [0, 2], [2, 0])
  dnfolder = f.split('/')[-1].split('.')[0]
  tifname = f.split('/')[-1].split('.')[0]+'.tif'
  tiffolder_path = os.path.join(datasets_path_processing, dnfolder)
  testfolder_path = os.path.join(tiffolder_path, dnfolder + '_*')
  if not os.path.exists(tiffolder_path):
    os.mkdir(tiffolder_path)
  tif_path = os.path.join(tiffolder_path, tifname)
  print(f)
  print(tif_path)
  tif.imwrite(tif_path, stack.astype('float'), photometric='minisblack')

  br = BgRemover(working_dir, tif_path, half_wid=12)
  br.draw_bg()
  br.show_bg()
  br.remove_bg()
  br.show_spectrum(fs=180)
  br.save_out()


  stack = io.imread(tif_path)
  Lt, Ly, Lx = stack.shape
  print(Lt)
  print(Ly)
  print(Lx)

  n_epochs = 2                # number of training epochs
  GPU = '0'                   # the index of GPU you will use (e.g. '0', '0,1', '0,1,2')
  train_datasets_size = np.max([4000, int(np.ceil(Lt/4))])  # datasets size for training (how many 3D patches)
  patch_x = int(np.ceil(Lx/4)) # was /4   # the width, height, and length of 3D patches (use isotropic patch size by default)
  patch_y = int(np.ceil(Ly/4)) # was /4
  patch_t = 300 #100
  overlap_factor = 0.4        # the overlap factor between two adjacent patches
  num_workers = 0             # if you use Windows system, set this to 0.

  # Setup some parameters for result visualization during training period (optional)
  save_test_images_per_epoch = True  # whether to save result images after each epoch

  train_dict = {
      # dataset dependent parameters
      'patch_x': patch_x,                          # the width of 3D patches
      'patch_y': patch_y,                          # the height of 3D patches
      'patch_t': patch_t,                          # the time dimension (frames) of 3D patches
      'overlap_factor':overlap_factor,               # the factor for image intensity scaling
      'scale_factor': 1,                             # the factor for image intensity scaling
      'select_img_num': train_datasets_size,                        # select the number of images used for training (use 2000 frames in colab)
      'train_datasets_size': train_datasets_size,    # datasets size for training (how many 3D patches)
      'datasets_path': tiffolder_path,                # folder containing files for training
      'pth_dir': tiffolder_path,                            # the path for pth file and result images

      # network related parameters
      'n_epochs': n_epochs,                          # the number of training epochs
      'lr': 0.00005,                                 # learning rate
      'b1': 0.5,                                     # Adam: bata1
      'b2': 0.999,                                   # Adam: bata2
      'fmap': 8,  # was 16 by default                                    # model complexity
      'GPU': GPU,                                    # GPU index
      'num_workers': num_workers,                    # if you use Windows system, set this to 0.
      'visualize_images_per_epoch': False,                       # whether to show result images after each epoch
      'save_test_images_per_epoch': save_test_images_per_epoch,  # whether to save result images after each epoch
      'colab_display': True
  }
  tc = training_class(train_dict)
  tc.run()

  test_datasize = Lt
  para_path_tmp = glob.glob(os.path.join(testfolder_path, '*.yaml'))[0]
  denoise_model = para_path_tmp.split('/')[-2] #datetime.datetime.now().strftime("%Y%m%dT%H%M%S") #'stackraw2_202210240402'
  para_path = glob.glob(os.path.join(tiffolder_path, denoise_model, '*.yaml'))[0]

  import yaml

  with open(para_path, "r") as stream:
      #para_dict = yaml.safe_load(stream)
      patch_t = train_dict['patch_t']
      patch_x = train_dict['patch_x']
      patch_y = train_dict['patch_y']
      fmap = train_dict['fmap']
      overlap_factor = train_dict['overlap_factor']

  test_dict = {
      # dataset dependent parameters
      'patch_x': patch_x,               # the width of 3D patches
      'patch_y': patch_y,               # the height of 3D patches
      'patch_t': patch_t,               # the time dimension (frames) of 3D patches
      'overlap_factor':overlap_factor,    # overlap factor
      'scale_factor': 1,                  # the factor for image intensity scaling
      'test_datasize': test_datasize,     # the number of frames to be tested
      'datasets_path': tiffolder_path,     # folder containing all files to be tested
      'pth_dir': tiffolder_path,                 # pth file root path
      'denoise_model' : denoise_model,    # A folder containing all models to be tested
      'output_dir' : tiffolder_path,         # result file root path
      # network related parameters
      'fmap': fmap,                         # number of feature maps
      'GPU': GPU,                         # GPU index
      'num_workers': num_workers,         # if you use Windows system, set this to 0.
      'visualize_images_per_epoch': False,# whether to display inference performance after each epoch
      'save_test_images_per_epoch': True, # whether to save inference image after each epoch in pth path
      'colab_display': True
  }

  tc = testing_class(test_dict)
  tc.run()

  if tc.colab_display:
      display_filename = tc.result_display
      print('\033[1;31mDisplaying denoised file of the last epoch-----> \033[0m')
      print(display_filename)
      # normalize the image and display
      img = display_img(display_filename,norm_min_percent=1, norm_max_percent=99)
      plt.imshow(img,cmap=plt.cm.gray,vmin=0,vmax=255)
      plt.axis('off')
      plt.show()

  outtiff_path = glob.glob(os.path.join(tiffolder_path, 'DataFolderIs_*', 'E_02_*', '*output.tif'))[0]
  outtiff_path_folder = outtiff_path.split('/')[-3] #datetime.datetime.now().strftime("%Y%m%dT%H%M%S") #'stackraw2_202210240402'
  outtiff_path_folder_pth = os.path.join(tiffolder_path, outtiff_path_folder)
  shutil.move(outtiff_path, datasets_path_complete)
  shutil.rmtree(outtiff_path_folder_pth)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU accessiable. Use GPU for computation.
GPU ID:  0 | A100-SXM4-40GB | Memory: 40536 MB
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2021 NVIDIA Corporation
Built on Sun_Feb_14_21:12:58_PST_2021
Cuda compilation tools, release 11.2, V11.2.152
Build cuda_11.2.r11.2/compiler.29618528_0
PyTorch version:  1.12.1+cu113
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
(256, 128, 5869)


NameError: ignored